# Sales Forecast Regression

**Regression Project 55**

Full pipeline: EDA → preprocessing → baseline → LazyPredict → FLAML → top-3 evaluation.

## 2. Project Overview

Predict item-level sales at BigMart outlets using product attributes (weight, visibility, MRP,
fat content, type) and outlet attributes (size, location, type, establishment year).

This is a classic retail regression problem that combines product and store features.
It teaches how to handle mixed feature types, clean inconsistent categories, and model
sales with practical business context.

## 3. Learning Objectives

1. Build a regression pipeline for retail sales prediction
2. Clean inconsistent categorical data (e.g., fat content labels)
3. Handle missing values in both numeric and categorical features
4. Engineer features combining product and outlet attributes
5. Compare baseline vs LazyPredict vs FLAML vs top-3 models
6. Evaluate with R², RMSE, MAE and residual analysis

## 4. Problem Statement

> Given product and outlet features, predict **Item_Outlet_Sales** for each product-store pair.

This is a **regression** task. Primary metric: **R²**; secondary: RMSE, MAE.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Retail managers | Inventory planning and stock allocation |
| Supply chain | Demand forecasting for procurement |
| Marketing | Identify high/low performing product-outlet combos |
| Finance | Revenue projections |
| ML learners | Mixed feature regression with real retail data |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Source | Kaggle: `brijbhushannanda1979/bigmart-sales-data` |
| Records | ~8,523 |
| Features | 11 (product + outlet attributes) |
| Target | `Item_Outlet_Sales` (continuous, sales value) |
| Key columns | Item_MRP, Outlet_Type, Outlet_Size, Item_Visibility, Item_Type |

## 7. Dataset Source and License Notes

- **Kaggle**: [brijbhushannanda1979/bigmart-sales-data](https://www.kaggle.com/datasets/brijbhushannanda1979/bigmart-sales-data)
- Originally from an Analytics Vidhya hackathon
- License: check Kaggle dataset page
- Downloaded via `kagglehub` at runtime

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['lazypredict', 'flaml', 'kagglehub', 'xgboost', 'lightgbm']:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
print('All imports loaded.')

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = 'brijbhushannanda1979/bigmart-sales-data'
TARGET = 'Item_Outlet_Sales'
TEST_SIZE = 0.15
VAL_SIZE = 0.15
FLAML_BUDGET = 120
MAX_ROWS = 50000

## 11. Dataset Download or Loading

Download the BigMart Sales dataset via `kagglehub`.

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(
        f'Dataset download failed: {e}\n'
        'Ensure KAGGLE_API_TOKEN is set. '
        'Fallback: download from https://www.kaggle.com/datasets/brijbhushannanda1979/bigmart-sales-data'
    )
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV files found in {path}'
# Prefer Train.csv
train_files = [f for f in csv_files if 'train' in os.path.basename(f).lower()]
chosen = train_files[0] if train_files else sorted(csv_files, key=os.path.getsize, reverse=True)[0]
df_raw = pd.read_csv(chosen)
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 12. Data Validation Checks

Verify target column, missing values, duplicates, and data consistency.

In [ ]:
assert TARGET in df_raw.columns, f"Target '{TARGET}' not found"
print(f'Missing values:\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}\n')
print(f'Duplicated rows: {df_raw.duplicated().sum()}')
df = df_raw.copy()
# Drop identifier columns
id_cols = [c for c in df.columns if 'identifier' in c.lower() or c.lower() in ['unnamed: 0']]
if id_cols:
    df = df.drop(columns=id_cols)
    print(f'Dropped ID columns: {id_cols}')
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Shape after cleaning: {df.shape}')
df.dtypes

## 13. Exploratory Data Analysis

Explore product and outlet feature distributions and their relationship with sales.

In [ ]:
df.describe()

In [ ]:
# Numeric feature distributions
num_feats = df.select_dtypes(include=[np.number]).columns.tolist()
num_feats_plot = [c for c in num_feats if c != TARGET][:4]
if num_feats_plot:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, col in zip(axes.flat, num_feats_plot):
        df[col].dropna().hist(bins=30, ax=ax, alpha=0.7, color='steelblue')
        ax.set_title(col)
    plt.tight_layout(); plt.show()

In [ ]:
# Sales by outlet type and item type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if 'Outlet_Type' in df.columns:
    df.groupby('Outlet_Type')[TARGET].mean().sort_values().plot.barh(ax=axes[0], color='coral')
    axes[0].set_title(f'Mean Sales by Outlet Type')
if 'Item_Type' in df.columns:
    df.groupby('Item_Type')[TARGET].mean().sort_values().plot.barh(ax=axes[1], color='seagreen')
    axes[1].set_title(f'Mean Sales by Item Type')
plt.tight_layout(); plt.show()

In [ ]:
# Check Item_Fat_Content inconsistency
if 'Item_Fat_Content' in df.columns:
    print('Fat content values BEFORE cleaning:')
    print(df['Item_Fat_Content'].value_counts())
    # Clean inconsistent labels
    df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
        'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular'
    })
    print('\nFat content values AFTER cleaning:')
    print(df['Item_Fat_Content'].value_counts())

In [ ]:
num_cols_eda = df.select_dtypes(include=[np.number]).columns.tolist()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[num_cols_eda].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout(); plt.show()

## 14. Target Analysis

Item-outlet sales are right-skewed with many low-sales items and a few high performers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df[TARGET], bins=50, color='coral', alpha=0.7)
axes[0].set_title(f'{TARGET} Distribution')
axes[1].hist(np.log1p(df[TARGET]), bins=50, color='seagreen', alpha=0.7)
axes[1].set_title(f'log({TARGET}) Distribution')
plt.tight_layout(); plt.show()
print(f'Mean: {df[TARGET].mean():,.0f}')
print(f'Median: {df[TARGET].median():,.0f}')
print(f'Skewness: {df[TARGET].skew():.2f}')

## 15. Train / Validation / Test Split

70/15/15 split with fixed seed.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

## 16. Preprocessing Strategy

- **Numeric** (Item_Weight, Item_Visibility, Item_MRP, Outlet_Establishment_Year): Impute median + scale
- **Categorical** (Item_Fat_Content, Item_Type, Outlet_Size, Outlet_Location_Type, Outlet_Type): Impute mode + OHE
- Note: `Item_Weight` has ~17% missing values — imputation by median is a reasonable strategy
- `Outlet_Size` has missing values — likely correlated with outlet type

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')

## 17. Feature Engineering

- **Outlet age**: Current year minus establishment year
- **Item category**: Broad category from first 2 chars of Item_Identifier (FD=Food, DR=Drink, NC=Non-Consumable)
- **MRP bin**: Discretized MRP for pattern discovery
- **Price tier**: Low/Medium/High based on MRP quartiles

In [ ]:
def engineer_features(d):
    d = d.copy()
    if 'Outlet_Establishment_Year' in d.columns:
        d['outlet_age'] = 2024 - d['Outlet_Establishment_Year']
    if 'Item_MRP' in d.columns:
        d['mrp_bin'] = pd.cut(d['Item_MRP'], bins=4, labels=False)
    # Clean fat content if not done already
    if 'Item_Fat_Content' in d.columns:
        d['Item_Fat_Content'] = d['Item_Fat_Content'].replace({
            'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular'
        })
    return d

X_train = engineer_features(X_train)
X_val = engineer_features(X_val)
X_test = engineer_features(X_test)
# Rebuild preprocessor
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
transformers = []
if num_cols:
    transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols))
if cat_cols:
    transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features after engineering: {X_train.shape[1]}')

## 18. Baseline Model

Three baselines: DummyRegressor (mean), LinearRegression, RandomForest.

In [ ]:
results = {}
for name, reg in [
    ('Dummy (mean)', DummyRegressor(strategy='mean')),
    ('LinearRegression', LinearRegression()),
    ('RandomForest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1))
]:
    pipe = Pipeline([('pre', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_val)
    r = {'RMSE': np.sqrt(mean_squared_error(y_val, yp)),
         'MAE': mean_absolute_error(y_val, yp),
         'R2': r2_score(y_val, yp)}
    results[name] = r
    print(f"{name}: R2={r['R2']:.4f}, RMSE={r['RMSE']:,.0f}, MAE={r['MAE']:,.0f}")

## 19. LazyPredict Benchmark

Quick comparison of ~30 regression models.

In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

FLAML with 2-minute budget for automated model selection.

In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='regression', metric='r2',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best estimator: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'RMSE': np.sqrt(mean_squared_error(y_val, yp_fl)),
                    'MAE': mean_absolute_error(y_val, yp_fl),
                    'R2': r2_score(y_val, yp_fl)}
print(f"FLAML: R2={results['FLAML']['R2']:.4f}, RMSE={results['FLAML']['RMSE']:,.0f}")

## 21. Top 3 Model Selection

Based on LazyPredict/FLAML results, we select three strong models.

In [ ]:
try:
    from lightgbm import LGBMRegressor; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBRegressor; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm:
    top3['LightGBM'] = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesRegressor
    top3['ExtraTrees'] = ExtraTreesRegressor(n_estimators=500, random_state=SEED, n_jobs=-1)
if has_xgb:
    top3['XGBoost'] = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1)
else:
    from sklearn.ensemble import AdaBoostRegressor
    top3['AdaBoost'] = AdaBoostRegressor(n_estimators=300, random_state=SEED)
top3['GradBoosting'] = GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')

## 22. Final Training and Evaluation of Top 3

Train on training set, evaluate on held-out test set.

In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
predictions = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    predictions[name] = yp
    final[name] = {'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
                   'MAE': mean_absolute_error(y_test, yp),
                   'R2': r2_score(y_test, yp)}
    print(f"{name}: R2={final[name]['R2']:.4f}, RMSE={final[name]['RMSE']:,.0f}, MAE={final[name]['MAE']:,.0f}")
pd.DataFrame(final).T.sort_values('R2', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    ax.scatter(y_test, yp, alpha=0.3, s=10)
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    ax.plot([mn, mx], [mn, mx], 'r--', lw=2)
    ax.set_xlabel('Actual Sales'); ax.set_ylabel('Predicted Sales')
    ax.set_title(f"{name} (R2={final[name]['R2']:.3f})")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(6*len(top3), 5))
if len(top3) == 1: axes = [axes]
for ax, (name, yp) in zip(axes, predictions.items()):
    residuals = y_test.values - yp
    ax.scatter(yp, residuals, alpha=0.3, s=10)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residual')
    ax.set_title(f'{name} Residuals')
plt.tight_layout(); plt.show()

## 23. Error Analysis

Where does the best model struggle most?

In [ ]:
best_name = max(final, key=lambda k: final[k]['R2'])
best_preds = predictions[best_name]
abs_errors = np.abs(y_test.values - best_preds)
print(f'Best model: {best_name}')
print(f'Mean absolute error: {abs_errors.mean():,.0f}')
print(f'Median absolute error: {np.median(abs_errors):,.0f}')
print(f'90th percentile error: {np.percentile(abs_errors, 90):,.0f}')
print(f'Max error: {abs_errors.max():,.0f}')

## 24. Interpretation and Business Insight

- **Item_MRP** is the strongest predictor — higher-priced items naturally generate higher sales values
- **Outlet_Type** (Supermarket Type 1/2/3 vs Grocery Store) significantly affects sales
- **Outlet age** matters — established outlets tend to have higher and more stable sales
- **Item_Visibility** has a counter-intuitive slight negative correlation (overexposed items may be lower-demand)
- **Item_Fat_Content** has minimal effect after accounting for MRP and outlet type
- The model captures product-outlet interaction patterns that simple rules cannot

## 25. Limitations

1. Item_Identifier was dropped — individual item identity could help
2. No temporal dimension — sales may be seasonal
3. Item_Weight has 17% missing values
4. Only 10 outlets — limited store diversity
5. No promotional/discount information
6. Dataset from a competition — may be synthetic or modified

## 26. How to Improve This Project

1. Target encode Item_Identifier for per-item baselines
2. Add interaction features (MRP × Outlet_Type)
3. Use log-transform on target for better residual behavior
4. Impute Item_Weight using item category means
5. Hyperparameter tuning with Optuna or Bayesian search

## 27. Production Considerations

- Integrate with inventory management systems
- Weekly/monthly retraining with new sales data
- Store-specific model variants
- Alert system for demand anomalies
- Dashboard for store managers showing predicted vs actual

## 28. Common Mistakes

1. Not cleaning Item_Fat_Content inconsistencies
2. Treating Item_Identifier as a feature (high cardinality, leakage risk)
3. Not handling missing Outlet_Size
4. Ignoring the strong MRP–sales relationship
5. Using accuracy metrics instead of regression metrics

## 29. Mini Challenge / Exercises

1. Try log-transforming Item_Outlet_Sales and compare R²
2. Impute Item_Weight using mean per Item_Type
3. Create a feature: is the outlet a grocery store vs supermarket
4. Compare models with and without Item_Visibility
5. Build a stacking ensemble combining the top 3 models

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Regression — predict item-outlet sales |
| Dataset | BigMart Sales (~8,523 records) |
| Difficulty | Medium |
| Key insight | Item_MRP and Outlet_Type dominate prediction |
| Best approach | Gradient boosting with outlet age + MRP features |
| Primary metric | R² |
| Baseline comparison | Tree models significantly outperform linear models |